In [ ]:
# =======================
# CONFIG
# =======================
SUBMISSION_TYPE = "label"   # "label" | "prob" | "onehot"

# =======================
# WARNINGS & IMPORTS
# =======================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# =======================
# LOAD DATA
# =======================
train = pd.read_csv("/kaggle/input/mse-2-ai-201-b-aiml-a/train.csv")
test  = pd.read_csv("/kaggle/input/mse-2-ai-201-b-aiml-a/test.csv")

test_ids = test["id"]

# =======================
# DATA CLEANING
# =======================
train.fillna(train.median(numeric_only=True), inplace=True)
test.fillna(test.median(numeric_only=True), inplace=True)

for col in train.select_dtypes(include="object"):
    train[col].fillna(train[col].mode()[0], inplace=True)
    if col in test.columns:
        test[col].fillna(test[col].mode()[0], inplace=True)

# =======================
# EDA (4–5 GRAPHS)
# =======================
sns.countplot(x="Class", data=train)
plt.title("Class Distribution")
plt.show()

train["Class"].value_counts().plot.pie(autopct="%1.1f%%")
plt.title("Class Distribution (Pie)")
plt.ylabel("")
plt.show()

sns.boxplot(data=train.select_dtypes(include=np.number))
plt.title("Outlier Analysis")
plt.show()

train.select_dtypes(include=np.number).hist(figsize=(10,6))
plt.suptitle("Feature Distributions")
plt.show()

sns.heatmap(train.select_dtypes(include=np.number).corr(), cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# =======================
# FEATURES & TARGET
# =======================
y = train["Class"]
X = train.drop(columns=["Class","id"], errors="ignore")
test = test.drop(columns=["id"], errors="ignore")

# =======================
# ENCODING FEATURES
# =======================
cat_cols = X.select_dtypes(include="object").columns
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])
test[cat_cols] = oe.transform(test[cat_cols])

# =======================
# ENCODE TARGET
# =======================
le = LabelEncoder()
y_enc = le.fit_transform(y)

# =======================
# ALIGN & SCALE
# =======================
X, test = X.align(test, axis=1, fill_value=0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test)

# =======================
# TRAIN / VALIDATION
# =======================
X_tr, X_val, y_tr, y_val = train_test_split(
    X_scaled, y_enc, test_size=0.2,
    stratify=y_enc, random_state=42
)

# =======================
# MODEL + TUNING
# =======================
rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(
    rf,
    param_grid={"n_estimators":[100,200]},
    cv=3,
    scoring="accuracy"
)

grid.fit(X_tr, y_tr)
best_model = grid.best_estimator_

# =======================
# EVALUATION
# =======================
val_pred = best_model.predict(X_val)

print("Accuracy :", accuracy_score(y_val, val_pred))
print("Precision:", precision_score(y_val, val_pred, average="weighted"))
print("Recall   :", recall_score(y_val, val_pred, average="weighted"))
print("F1 Score :", f1_score(y_val, val_pred, average="weighted"))
print("\nClassification Report:\n",
      classification_report(y_val, val_pred))

sns.heatmap(confusion_matrix(y_val, val_pred),
            annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

# =======================
# TEST PREDICTION
# =======================
proba = best_model.predict_proba(test_scaled)

# =======================
# SUBMISSION HANDLING
# =======================
if SUBMISSION_TYPE == "label":
    preds = le.inverse_transform(np.argmax(proba, axis=1))
    submission = pd.DataFrame({
        "id": test_ids,
        "Status": preds
    })

elif SUBMISSION_TYPE == "prob":
    submission = pd.DataFrame(
        proba, columns=le.classes_
    )
    submission.insert(0, "id", test_ids)

elif SUBMISSION_TYPE == "onehot":
    one_hot = (proba == proba.max(axis=1, keepdims=True)).astype(int)
    submission = pd.DataFrame(
        one_hot, columns=le.classes_
    )
    submission.insert(0, "id", test_ids)

else:
    raise ValueError("Invalid SUBMISSION_TYPE")

# =======================
# SAVE FILE
# =======================
submission.to_csv("submission.csv", index=False)
print("submission.csv generated successfully")